# RAG Document Q&A System
## Reinforcement Learning Research Papers

A RAG-powered Q&A system built on 10 reinforcement learning research papers.

**Stack:**
- **LLM:** GLM-5 (cloud) via Ollama
- **Embeddings:** Qwen3-Embedding-8B via Ollama (4096 dimensions)
- **Vector DB:** ChromaDB
- **Framework:** LangChain

**Stretch Goals:** A (Chunk Comparison) | B (Hybrid BM25) | C (Metadata Filtering) | D (Streamlit UI) | E (Multi-document)

In [1]:
# Imports
import os
import json
import warnings
from pathlib import Path
from datetime import datetime

from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader, TextLoader, CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

warnings.filterwarnings('ignore')

DATA_DIR = "data"
CHROMA_DIR = "./chroma_db"

print("All imports successful.")

All imports successful.


---
## Step 1: Load Documents

Loading 10 reinforcement learning research papers (PDFs) using LangChain's `PyPDFLoader` via `DirectoryLoader`.

**Stretch Goal E (Multi-document):** We also create a summary text file and a CSV metadata file to demonstrate loading 3+ different document types into the same vector store.

In [2]:
# Stretch Goal E: Create additional document types for multi-document loading

# 1. Summary TXT file
summary_text = """Reinforcement Learning Research Papers - Summary Index

This collection contains 10 research papers covering key topics in reinforcement learning:

1. Deep Recurrent Q-Learning for Partially Observable MDPs (Hausknecht & Stone, 2015)
   - Combines LSTM with DQN to handle partial observability in Atari games.

2. General Value Function Networks (Schlegel et al., 2021)
   - Proposes architectures for learning predictive knowledge using GVFs.

3. Recurrent Model-Free RL Can Be a Strong Baseline for Many POMDPs (Ni et al., 2021)
   - Shows that simple recurrent model-free methods match or beat specialized POMDP algorithms.

4. Recurrent Experience Replay in Distributed Reinforcement Learning (Kapturowski et al., 2019)
   - Introduces R2D2 agent with burn-in strategy for training recurrent RL agents at scale.

5. Stabilizing Transformers for Reinforcement Learning (Parisotto et al., 2020)
   - Proposes Gated Transformer-XL (GTrXL) architecture for stable RL training.

6. Reward Machines: Exploiting Reward Function Structure in RL (Icarte et al., 2022)
   - Introduces reward machines as a formalism for structured reward specification.

7. On Overfitting and Asymptotic Bias in Batch RL with Partial Observability (Francois-Lavet et al., 2019)
   - Analyzes overfitting and bias issues when training batch RL with limited observations.

8. Constrained Policy Optimization (Achiam et al., 2017)
   - Proposes CPO algorithm for safe RL with constraint satisfaction guarantees.

9. Benchmarking Batch Deep Reinforcement Learning Algorithms (Fujimoto et al., 2019)
   - Comprehensive evaluation of off-policy batch deep RL methods.

10. Mastering Diverse Domains through World Models - DreamerV3 (Hafner et al., 2023)
    - A single RL agent that masters 150+ diverse tasks without domain-specific tuning.

Key themes: POMDPs, batch/offline RL, safe RL, memory architectures, world models, reward shaping.
"""

with open(os.path.join(DATA_DIR, "papers_summary.txt"), "w") as f:
    f.write(summary_text)

# 2. CSV metadata file
csv_content = """paper_id,title,authors,year,venue,topic
1,Deep Recurrent Q-Learning for Partially Observable MDPs,Hausknecht and Stone,2015,AAAI Workshop,POMDP
2,General Value Function Networks,Schlegel et al.,2021,JAIR,Predictive Knowledge
3,Recurrent Model-Free RL Can Be a Strong Baseline for Many POMDPs,Ni et al.,2021,NeurIPS,POMDP
4,Recurrent Experience Replay in Distributed Reinforcement Learning,Kapturowski et al.,2019,ICLR,Distributed RL
5,Stabilizing Transformers for Reinforcement Learning,Parisotto et al.,2020,ICML,Memory Architecture
6,Reward Machines: Exploiting Reward Function Structure in RL,Icarte et al.,2022,JAIR,Reward Shaping
7,On Overfitting and Asymptotic Bias in Batch RL with Partial Observability,Francois-Lavet et al.,2019,JAIR,Batch RL
8,Constrained Policy Optimization,Achiam et al.,2017,ICML,Safe RL
9,Benchmarking Batch Deep Reinforcement Learning Algorithms,Fujimoto et al.,2019,arXiv,Batch RL
10,Mastering Diverse Domains through World Models,Hafner et al.,2023,arXiv,World Models
"""

with open(os.path.join(DATA_DIR, "papers_metadata.csv"), "w") as f:
    f.write(csv_content)

print("Created papers_summary.txt and papers_metadata.csv for multi-document loading (Stretch Goal E).")

Created papers_summary.txt and papers_metadata.csv for multi-document loading (Stretch Goal E).


In [3]:
# Load PDFs
pdf_loader = DirectoryLoader(
    DATA_DIR,
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True
)
pdf_documents = pdf_loader.load()
print(f"PDF documents loaded: {len(pdf_documents)} pages")

# Load TXT (Stretch Goal E)
txt_loader = TextLoader(os.path.join(DATA_DIR, "papers_summary.txt"), encoding="utf-8")
txt_documents = txt_loader.load()
print(f"TXT documents loaded: {len(txt_documents)}")

# Load CSV (Stretch Goal E)
csv_loader = CSVLoader(os.path.join(DATA_DIR, "papers_metadata.csv"))
csv_documents = csv_loader.load()
print(f"CSV documents loaded: {len(csv_documents)} rows")

# Combine all documents
all_documents = pdf_documents + txt_documents + csv_documents
print(f"\nTotal documents loaded: {len(all_documents)}")
print(f"  - PDFs: {len(pdf_documents)} pages")
print(f"  - TXT:  {len(txt_documents)} documents")
print(f"  - CSV:  {len(csv_documents)} rows")
print(f"\nDocument types loaded: PDF, TXT, CSV (Stretch Goal E: Multi-document ✅)")

  0%|          | 0/10 [00:00<?, ?it/s]

 10%|█         | 1/10 [00:00<00:03,  2.83it/s]

 20%|██        | 2/10 [00:03<00:14,  1.76s/it]

 30%|███       | 3/10 [00:03<00:07,  1.11s/it]

 40%|████      | 4/10 [00:11<00:22,  3.67s/it]

...

Impossible to decode XFormObject /arial-minus: '/arial-minus'


 90%|█████████ | 9/10 [00:12<00:00,  1.18it/s]

100%|██████████| 10/10 [00:15<00:00,  1.27s/it]

100%|██████████| 10/10 [00:15<00:00,  1.50s/it]

PDF documents loaded: 247 pages
TXT documents loaded: 1
CSV documents loaded: 10 rows

Total documents loaded: 258
  - PDFs: 247 pages
  - TXT:  1 documents
  - CSV:  10 rows

Document types loaded: PDF, TXT, CSV (Stretch Goal E: Multi-document ✅)


In [4]:
# Sample first document content
print("=" * 60)
print("SAMPLE: First PDF page")
print("=" * 60)
print(f"Source: {pdf_documents[0].metadata.get('source', 'unknown')}")
print(f"Content (first 500 chars):\n{pdf_documents[0].page_content[:500]}")
print(f"\nMetadata: {pdf_documents[0].metadata}")

SAMPLE: First PDF page
Source: data\01_deep_recurrent_q_learning_pomdps.pdf
Content (first 500 chars):
Deep Recurrent Q-Learning for Partially Observable MDPs
Matthew Hausknecht and Peter Stone
Department of Computer Science
The University of Texas at Austin
{mhauskn, pstone}@cs.utexas.edu
Abstract
Deep Reinforcement Learning has yielded proﬁcient
controllers for complex tasks. However, these con-
trollers have limited memory and rely on being able
to perceive the complete game screen at each deci-
sion point. To address these shortcomings, this arti-
cle investigates the effects of adding recurr

Metadata: {'producer': 'pdfTeX-1.40.12', 'creator': 'TeX', 'creationdate': '2017-01-13T01:19:41+00:00', 'moddate': '2017-01-13T01:19:41+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.1415926-2.3-1.40.12 (TeX Live 2011) kpathsea version 6.0.1', 'title': 'Deep Recurrent Q-Learning for Partially Observable MDPs', 'trapped': '/False', 'source': 'data\\01_deep_recurrent_q_learning_pomdps.pd

---
## Step 2: Chunk Documents

Using `RecursiveCharacterTextSplitter` with 3 different chunk sizes: **300, 500, 1000**.

This satisfies the base requirement (2 sizes) and **Stretch Goal A** (3 sizes for comparison).

In [5]:
def chunk_documents(docs, chunk_size=500, chunk_overlap=50):
    """Split documents into chunks and print stats."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    chunks = splitter.split_documents(docs)
    lengths = [len(c.page_content) for c in chunks]
    
    print(f"Chunk size: {chunk_size} | Overlap: {chunk_overlap}")
    print(f"  Total chunks: {len(chunks)}")
    print(f"  Smallest chunk: {min(lengths)} chars")
    print(f"  Largest chunk:  {max(lengths)} chars")
    print(f"  Average chunk:  {sum(lengths)/len(lengths):.0f} chars")
    print()
    return chunks

print("Chunking with 3 different sizes:")
print("=" * 60)

chunks_300 = chunk_documents(all_documents, chunk_size=300, chunk_overlap=30)
chunks_500 = chunk_documents(all_documents, chunk_size=500, chunk_overlap=50)
chunks_1000 = chunk_documents(all_documents, chunk_size=1000, chunk_overlap=100)

Chunking with 3 different sizes:
Chunk size: 300 | Overlap: 30
  Total chunks: 2944
  Smallest chunk: 2 chars
  Largest chunk:  300 chars
  Average chunk:  252 chars

Chunk size: 500 | Overlap: 50
  Total chunks: 1816
  Smallest chunk: 3 chars
  Largest chunk:  500 chars
  Average chunk:  412 chars

Chunk size: 1000 | Overlap: 100
  Total chunks: 953
  Smallest chunk: 36 chars
  Largest chunk:  1000 chars
  Average chunk:  825 chars



In [6]:
# Observations on chunking
print("CHUNK SIZE OBSERVATIONS")
print("=" * 60)
print(f"chunk_size=300 -> {len(chunks_300)} chunks (very granular, may lose context)")
print(f"chunk_size=500 -> {len(chunks_500)} chunks (balanced, good for most queries)")
print(f"chunk_size=1000 -> {len(chunks_1000)} chunks (fewer chunks, more context per chunk)")
print()
print("Sample chunk (size=500, chunk #0):")
print("-" * 40)
print(chunks_500[0].page_content[:300])
print("...")

CHUNK SIZE OBSERVATIONS
chunk_size=300 -> 2944 chunks (very granular, may lose context)
chunk_size=500 -> 1816 chunks (balanced, good for most queries)
chunk_size=1000 -> 953 chunks (fewer chunks, more context per chunk)

Sample chunk (size=500, chunk #0):
----------------------------------------
Deep Recurrent Q-Learning for Partially Observable MDPs
Matthew Hausknecht and Peter Stone
Department of Computer Science
The University of Texas at Austin
{mhauskn, pstone}@cs.utexas.edu
Abstract
Deep Reinforcement Learning has yielded proﬁcient
controllers for complex tasks. However, these con-
tr
...


---
## Step 3: Embed + Store in ChromaDB

Using **Qwen3-Embedding-8B** via Ollama (4096 dimensions, #1 on MTEB multilingual leaderboard).

Creating separate ChromaDB collections for each chunk size.

**Stretch Goal C (Metadata Filtering):** Adding structured metadata (source filename, page number, paper topic, year) to each chunk.

In [7]:
# Stretch Goal C: Enrich metadata before embedding
PAPER_METADATA = {
    "01_deep_recurrent_q_learning_pomdps": {"topic": "POMDP", "year": "2015", "venue": "AAAI Workshop"},
    "02_general_value_function_networks": {"topic": "Predictive Knowledge", "year": "2021", "venue": "JAIR"},
    "03_recurrent_model_free_rl_pomdps": {"topic": "POMDP", "year": "2021", "venue": "NeurIPS"},
    "04_recurrent_experience_replay_distributed_rl": {"topic": "Distributed RL", "year": "2019", "venue": "ICLR"},
    "05_stabilizing_transformers_rl": {"topic": "Memory Architecture", "year": "2020", "venue": "ICML"},
    "06_reward_machines": {"topic": "Reward Shaping", "year": "2022", "venue": "JAIR"},
    "07_overfitting_asymptotic_bias_batch_rl": {"topic": "Batch RL", "year": "2019", "venue": "JAIR"},
    "08_constrained_policy_optimization": {"topic": "Safe RL", "year": "2017", "venue": "ICML"},
    "09_benchmarking_batch_deep_rl": {"topic": "Batch RL", "year": "2019", "venue": "arXiv"},
    "10_mastering_diverse_domains_world_models": {"topic": "World Models", "year": "2023", "venue": "arXiv"},
    "papers_summary": {"topic": "Summary", "year": "2025", "venue": "N/A"},
    "papers_metadata": {"topic": "Metadata", "year": "2025", "venue": "N/A"},
}

def enrich_metadata(chunks):
    """Add topic, year, venue metadata to chunks based on source filename."""
    for chunk in chunks:
        source = chunk.metadata.get("source", "")
        filename = Path(source).stem
        for key, meta in PAPER_METADATA.items():
            if key in filename:
                chunk.metadata.update(meta)
                break
    return chunks

chunks_300 = enrich_metadata(chunks_300)
chunks_500 = enrich_metadata(chunks_500)
chunks_1000 = enrich_metadata(chunks_1000)

print("Metadata enrichment complete (Stretch Goal C ✅)")
print(f"Sample metadata: {chunks_500[0].metadata}")

Metadata enrichment complete (Stretch Goal C ✅)
Sample metadata: {'producer': 'pdfTeX-1.40.12', 'creator': 'TeX', 'creationdate': '2017-01-13T01:19:41+00:00', 'moddate': '2017-01-13T01:19:41+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.1415926-2.3-1.40.12 (TeX Live 2011) kpathsea version 6.0.1', 'title': 'Deep Recurrent Q-Learning for Partially Observable MDPs', 'trapped': '/False', 'source': 'data\\01_deep_recurrent_q_learning_pomdps.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'topic': 'POMDP', 'year': '2015', 'venue': 'AAAI Workshop'}


In [8]:
# Initialize embedding model
embedding_model = OllamaEmbeddings(model="qwen3-embedding")

# Test embedding
test_embedding = embedding_model.embed_query("reinforcement learning")
print(f"Embedding model: qwen3-embedding (Qwen3-Embedding-8B)")
print(f"Embedding dimensions: {len(test_embedding)}")
print(f"Sample values: {test_embedding[:5]}")

Embedding model: qwen3-embedding (Qwen3-Embedding-8B)
Embedding dimensions: 4096
Sample values: [0.030274063, 0.025110206, 0.0018374281, -0.022360569, 0.07703945]


In [9]:
# Clear any existing ChromaDB data for a clean run
import shutil
if os.path.exists(CHROMA_DIR):
    shutil.rmtree(CHROMA_DIR)
    print("Cleared existing ChromaDB data.")

# Create vector stores for each chunk size
print("\nEmbedding and storing chunks (this may take a few minutes)...")

print("\n[1/3] Embedding chunk_size=500 (primary)...")
vectorstore_500 = Chroma.from_documents(
    documents=chunks_500,
    embedding=embedding_model,
    collection_name="chunks_500",
    persist_directory=CHROMA_DIR
)
print(f"  ✅ Stored {len(chunks_500)} chunks")

print("\n[2/3] Embedding chunk_size=300...")
vectorstore_300 = Chroma.from_documents(
    documents=chunks_300,
    embedding=embedding_model,
    collection_name="chunks_300",
    persist_directory=CHROMA_DIR
)
print(f"  ✅ Stored {len(chunks_300)} chunks")

print("\n[3/3] Embedding chunk_size=1000...")
vectorstore_1000 = Chroma.from_documents(
    documents=chunks_1000,
    embedding=embedding_model,
    collection_name="chunks_1000",
    persist_directory=CHROMA_DIR
)
print(f"  ✅ Stored {len(chunks_1000)} chunks")

print("\n✅ All vector stores created successfully.")

Cleared existing ChromaDB data.

Embedding and storing chunks (this may take a few minutes)...

[1/3] Embedding chunk_size=500 (primary)...


  ✅ Stored 1816 chunks

[2/3] Embedding chunk_size=300...


  ✅ Stored 2944 chunks

[3/3] Embedding chunk_size=1000...


  ✅ Stored 953 chunks

✅ All vector stores created successfully.


---
## Step 4: Test Retrieval (BEFORE wiring up the LLM!)

Running 3 test queries using `similarity_search` and annotating relevance.

In [10]:
test_queries = [
    "How does DRQN handle partial observability in Atari games?",
    "What is the burn-in technique used in R2D2 for recurrent experience replay?",
    "How does DreamerV3 achieve generalization across diverse domains without tuning?",
]

print("RETRIEVAL TEST (Vector Search, chunk_size=500)")
print("=" * 70)

for i, query in enumerate(test_queries):
    print(f"\n{'='*70}")
    print(f"Query {i+1}: {query}")
    print(f"{'='*70}")
    
    results = vectorstore_500.similarity_search(query, k=3)
    
    for j, doc in enumerate(results):
        print(f"\n--- Retrieved Chunk {j+1} ---")
        print(f"Source: {doc.metadata.get('source', 'unknown')}")
        print(f"Topic:  {doc.metadata.get('topic', 'unknown')} | Year: {doc.metadata.get('year', 'unknown')}")
        print(f"Content (first 300 chars):")
        print(f"{doc.page_content[:300]}...")
    
    print(f"\n>> Relevance annotation for Query {i+1}: [TO BE FILLED AFTER RUNNING]")

RETRIEVAL TEST (Vector Search, chunk_size=500)

Query 1: How does DRQN handle partial observability in Atari games?

--- Retrieved Chunk 1 ---
Source: data\01_deep_recurrent_q_learning_pomdps.pdf
Topic:  POMDP | Year: 2015
Content (first 300 chars):
servability, and that when trained with full observations and
evaluated with partial observations, DRQN better handles
arXiv:1507.06527v4  [cs.LG]  11 Jan 2017...

--- Retrieved Chunk 2 ---
Source: data\01_deep_recurrent_q_learning_pomdps.pdf
Topic:  POMDP | Year: 2015
Content (first 300 chars):
POMDPs by combining a Long Short Term Memory with a
Deep Q-Network. The resulting Deep Recurrent Q-Network
(DRQN), despite seeing only a single frame at each step, is
still capable integrating information across frames to detect
relevant information such as velocity of on-screen objects.
Additionall...

--- Retrieved Chunk 3 ---
Source: data\01_deep_recurrent_q_learning_pomdps.pdf
Topic:  POMDP | Year: 2015
Content (first 300 chars):
cle investigate


--- Retrieved Chunk 1 ---
Source: data\04_recurrent_experience_replay_distributed_rl.pdf
Topic:  Distributed RL | Year: 2019
Content (first 300 chars):
tasks in a benchmark simultaneously while maintaining human-level performance.
2.3 T HE RECURRENT REPLAY DISTRIBUTED DQN A GENT
We propose a new agent, the Recurrent Replay Distributed DQN (R2D2), and use it to study the
interplay between recurrent state, experience replay, and distributed training....

--- Retrieved Chunk 2 ---
Source: data\04_recurrent_experience_replay_distributed_rl.pdf
Topic:  Distributed RL | Year: 2019
Content (first 300 chars):
(Mnih et al., 2015). A full list of hyper-parameters is provided in the Appendix.
We train the R2D2 agent with a single GPU-based learner, performing approximately 5 network up-
dates per second (each update on a mini-batch of64 length-80 sequences), and each actor performing
∼ 260 environment steps...

--- Retrieved Chunk 3 ---
Source: data\04_recurrent_experience_replay_distributed_rl.

In [11]:
# Stretch Goal C: Test metadata-filtered retrieval
print("METADATA FILTERED RETRIEVAL TEST (Stretch Goal C)")
print("=" * 70)

# Filter: only retrieve from POMDP papers
print("\nFilter: topic = 'POMDP'")
print("Query: How do recurrent networks help with partial observability?")
print("-" * 40)

filtered_results = vectorstore_500.similarity_search(
    "How do recurrent networks help with partial observability?",
    k=3,
    filter={"topic": "POMDP"}
)

for j, doc in enumerate(filtered_results):
    print(f"\n--- Chunk {j+1} ---")
    print(f"Source: {doc.metadata.get('source', 'unknown')}")
    print(f"Topic: {doc.metadata.get('topic', 'unknown')}")
    print(f"Content: {doc.page_content[:200]}...")

print("\n>> All results correctly filtered to POMDP papers ✅")

# Filter: only retrieve from 2019+ papers
print("\n" + "=" * 70)
print("Filter: year = '2023'")
print("Query: What world model architecture is used for multi-task RL?")
print("-" * 40)

filtered_results_2 = vectorstore_500.similarity_search(
    "What world model architecture is used for multi-task RL?",
    k=3,
    filter={"year": "2023"}
)

for j, doc in enumerate(filtered_results_2):
    print(f"\n--- Chunk {j+1} ---")
    print(f"Source: {doc.metadata.get('source', 'unknown')}")
    print(f"Year: {doc.metadata.get('year', 'unknown')}")
    print(f"Content: {doc.page_content[:200]}...")

print("\nMetadata filtering works correctly (Stretch Goal C ✅)")

METADATA FILTERED RETRIEVAL TEST (Stretch Goal C)

Filter: topic = 'POMDP'
Query: How do recurrent networks help with partial observability?
----------------------------------------

--- Chunk 1 ---
Source: data\01_deep_recurrent_q_learning_pomdps.pdf
Topic: POMDP
Content: trained with partial observations and evaluated with in-
crementally more complete observations, DRQN’s per-
formance scales as a function of observability. Con-
versely, when trained with full observ...

--- Chunk 2 ---
Source: data\03_recurrent_model_free_rl_pomdps.pdf
Topic: POMDP
Content: arXiv:1806.10729, 2018. 4
Kaelbling, L. P., Littman, M. L., and Cassandra, A. R. Planning
and acting in partially observable stochastic domains. Artif.
Intell., 1998. 3...

--- Chunk 3 ---
Source: data\01_deep_recurrent_q_learning_pomdps.pdf
Topic: POMDP
Content: Deep Recurrent Q-Learning for Partially Observable MDPs
Matthew Hausknecht and Peter Stone
Department of Computer Science
The University of Texas at Austin
{mhauskn, ps


--- Chunk 1 ---
Source: data\10_mastering_diverse_domains_world_models.pdf
Year: 2023
Content: Mastering Diverse Domains through World Models
Danijar Hafner,12 Jurgis Pasukonis,1 Jimmy Ba,2 Timothy Lillicrap1
Abstract
Developing a general algorithm that learns to solve tasks across a wide range...

--- Chunk 2 ---
Source: data\10_mastering_diverse_domains_world_models.pdf
Year: 2023
Content: over 150 tasks but also learns robustly across varying data and compute budgets, moving reinforce-
ment learning toward a wide range of practical applications. Applied out of the box, Dreamer is
the f...

--- Chunk 3 ---
Source: data\10_mastering_diverse_domains_world_models.pdf
Year: 2023
Content: T = 0
Model
5
 10
 15
 20
 25
 30
 35
 40
 45
 50
Figure 7: Multi-step predictions on Minecraft. The world model receives the first 5 frames as
context input and the predicts 45 steps into the future ...

Metadata filtering works correctly (Stretch Goal C ✅)


In [12]:
# Stretch Goal B: Hybrid BM25 + Vector retrieval test
print("HYBRID RETRIEVAL TEST - BM25 + Vector (Stretch Goal B)")
print("=" * 70)

# Set up BM25 retriever on the same chunks
bm25_retriever = BM25Retriever.from_documents(chunks_500)
bm25_retriever.k = 3

vector_retriever = vectorstore_500.as_retriever(search_kwargs={"k": 3})

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.4, 0.6]  # 40% keyword, 60% semantic
)

print("\nComparing Vector-only vs Hybrid for each test query:")
print()

for i, query in enumerate(test_queries):
    print(f"\n{'='*70}")
    print(f"Query {i+1}: {query}")
    
    # Vector-only
    vector_results = vectorstore_500.similarity_search(query, k=3)
    vector_sources = [Path(d.metadata.get('source', '')).stem for d in vector_results]
    
    # Hybrid
    hybrid_results = ensemble_retriever.invoke(query)
    hybrid_sources = [Path(d.metadata.get('source', '')).stem for d in hybrid_results[:3]]
    
    print(f"  Vector-only sources: {vector_sources}")
    print(f"  Hybrid sources:      {hybrid_sources}")
    print(f"  Same results? {vector_sources == hybrid_sources}")

print("\nHybrid search setup complete (Stretch Goal B ✅)")

HYBRID RETRIEVAL TEST - BM25 + Vector (Stretch Goal B)

Comparing Vector-only vs Hybrid for each test query:


Query 1: How does DRQN handle partial observability in Atari games?


  Vector-only sources: ['01_deep_recurrent_q_learning_pomdps', '01_deep_recurrent_q_learning_pomdps', '01_deep_recurrent_q_learning_pomdps']
  Hybrid sources:      ['01_deep_recurrent_q_learning_pomdps', '01_deep_recurrent_q_learning_pomdps', '01_deep_recurrent_q_learning_pomdps']
  Same results? True

Query 2: What is the burn-in technique used in R2D2 for recurrent experience replay?
  Vector-only sources: ['04_recurrent_experience_replay_distributed_rl', '04_recurrent_experience_replay_distributed_rl', '04_recurrent_experience_replay_distributed_rl']
  Hybrid sources:      ['04_recurrent_experience_replay_distributed_rl', '04_recurrent_experience_replay_distributed_rl', '04_recurrent_experience_replay_distributed_rl']
  Same results? True

Query 3: How does DreamerV3 achieve generalization across diverse domains without tuning?


  Vector-only sources: ['10_mastering_diverse_domains_world_models', '10_mastering_diverse_domains_world_models', '10_mastering_diverse_domains_world_models']
  Hybrid sources:      ['10_mastering_diverse_domains_world_models', '10_mastering_diverse_domains_world_models', '10_mastering_diverse_domains_world_models']
  Same results? True

Hybrid search setup complete (Stretch Goal B ✅)


---
## Step 5: Build the RAG Chain

Wiring up `RetrievalQA` with GLM-5 (cloud) via Ollama and a custom prompt template.

In [13]:
# Initialize LLM
llm = ChatOllama(
    model="glm-5:cloud",
    temperature=0,
)

# Custom prompt template
custom_prompt = PromptTemplate(
    template="""You are a helpful research assistant that answers questions about reinforcement learning papers based ONLY on the provided context.

Rules:
- Answer based ONLY on the provided context. Do not use external knowledge.
- If the context does not contain enough information, say "I don't have enough information in the provided context to answer this question."
- Cite which paper/source the information comes from when possible.
- Be concise but thorough.

Context:
{context}

Question: {question}

Answer:""",
    input_variables=["context", "question"]
)

# Build RAG chain
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore_500.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": custom_prompt}
)

print("RAG chain built successfully.")
print(f"  LLM: glm-5:cloud")
print(f"  Retriever: ChromaDB (chunk_size=500)")
print(f"  Chain type: stuff")

RAG chain built successfully.
  LLM: glm-5:cloud
  Retriever: ChromaDB (chunk_size=500)
  Chain type: stuff


In [14]:
# Run the same 3 test queries through the full RAG chain
print("RAG CHAIN RESULTS")
print("=" * 70)

for i, query in enumerate(test_queries):
    print(f"\n{'='*70}")
    print(f"Query {i+1}: {query}")
    print(f"{'='*70}")
    
    result = rag_chain.invoke({"query": query})
    
    print(f"\nAnswer:\n{result['result']}")
    print(f"\nSources:")
    for doc in result['source_documents']:
        print(f"  - {Path(doc.metadata.get('source', 'unknown')).stem} "
              f"(topic: {doc.metadata.get('topic', 'N/A')}, "
              f"year: {doc.metadata.get('year', 'N/A')})")

RAG CHAIN RESULTS

Query 1: How does DRQN handle partial observability in Atari games?



Answer:
Based on the provided context, DRQN handles partial observability in the following ways:

*   **Architecture:** It combines a Long Short Term Memory (LSTM) with a Deep Q-Network (DQN) by replacing the first post-convolutional fully-connected layer with a recurrent LSTM (arXiv:1507.06527v4).
*   **Temporal Integration:** Despite observing only a single frame at each timestep, DRQN integrates information across frames to detect relevant details, such as the velocity of on-screen objects (arXiv:1507.06527v4).
*   **Flickering Screens:** It is better equipped than a standard DQN to handle partial observability caused by flickering game screens, successfully replicating DQN performance on partially observed equivalents of Atari games (arXiv:1507.06527v4).

Sources:
  - 01_deep_recurrent_q_learning_pomdps (topic: POMDP, year: 2015)
  - 01_deep_recurrent_q_learning_pomdps (topic: POMDP, year: 2015)
  - 01_deep_recurrent_q_learning_pomdps (topic: POMDP, year: 2015)

Query 2: What is t


Answer:
I don't have enough information in the provided context to answer this question. The text describes the R2D2 agent and its setup (e.g., similarity to Ape-X, use of n-step double Q-learning, and training speeds), but it does not define or mention the "burn-in" technique.

Sources:
  - 04_recurrent_experience_replay_distributed_rl (topic: Distributed RL, year: 2019)
  - 04_recurrent_experience_replay_distributed_rl (topic: Distributed RL, year: 2019)
  - 04_recurrent_experience_replay_distributed_rl (topic: Distributed RL, year: 2019)

Query 3: How does DreamerV3 achieve generalization across diverse domains without tuning?



Answer:
Based on the provided context from the DreamerV3 paper, the algorithm achieves generalization across diverse domains with a single configuration through the following mechanisms:

*   **Robustness Techniques:** The text states that techniques based on "normalization, balancing, and transformations enable stable learning across domains." Specific implementations include observation symlog, KL balance and free bits, 1% unimix for all categorical variables, percentile return normalization, and a symexp twohot loss for the reward head and critic.
*   **World Model Learning:** Dreamer learns a model of the environment and improves its behavior by "imagining future scenarios."
*   **Architecture and Optimizer:** It employs a Block GRU architecture with RMSNorm normalization and SiLu activation. The optimizer uses Adaptive gradient clipping (AGC) and LaProp (RMSProp followed by momentum).
*   **Replay Buffer:** It utilizes a larger capacity replay buffer with an online queue that sto

---
## Step 6: Evaluate

5-question evaluation set with 3 metrics:
- **Retrieval:** Did it find the right chunks?
- **Faithfulness:** Is the answer grounded in context (not hallucinated)?
- **Correctness:** Is the answer actually right?

In [15]:
eval_set = [
    {
        "question": "What architecture does DRQN use to handle partial observability?",
        "expected_answer": "DRQN replaces the first fully connected layer of DQN with an LSTM recurrent layer to handle partial observability.",
        "expected_source_keyword": "deep_recurrent_q_learning"
    },
    {
        "question": "What is the burn-in strategy in R2D2?",
        "expected_answer": "Burn-in uses a portion of the replay sequence to initialize the recurrent state before the actual training segment, producing a better initial hidden state.",
        "expected_source_keyword": "recurrent_experience_replay"
    },
    {
        "question": "What is the key contribution of Constrained Policy Optimization (CPO)?",
        "expected_answer": "CPO provides near-constraint satisfaction guarantees at each policy update, enabling safe reinforcement learning with cost constraints.",
        "expected_source_keyword": "constrained_policy_optimization"
    },
    {
        "question": "What is the Gated Transformer-XL (GTrXL) and what problem does it solve?",
        "expected_answer": "GTrXL is a stabilized transformer architecture for RL that replaces residual connections with gating layers, enabling stable training of transformers in RL settings.",
        "expected_source_keyword": "stabilizing_transformers"
    },
    {
        "question": "How does DreamerV3 handle the challenge of varying signal magnitudes across different domains?",
        "expected_answer": "DreamerV3 uses symlog predictions that transform targets with a logarithmic function to handle the wide range of reward magnitudes across different domains.",
        "expected_source_keyword": "mastering_diverse_domains"
    },
]

print(f"Evaluation set: {len(eval_set)} questions")
for i, item in enumerate(eval_set):
    print(f"  Q{i+1}: {item['question'][:70]}...")

Evaluation set: 5 questions
  Q1: What architecture does DRQN use to handle partial observability?...
  Q2: What is the burn-in strategy in R2D2?...
  Q3: What is the key contribution of Constrained Policy Optimization (CPO)?...
  Q4: What is the Gated Transformer-XL (GTrXL) and what problem does it solv...
  Q5: How does DreamerV3 handle the challenge of varying signal magnitudes a...


In [16]:
# Run evaluation
def run_evaluation(chain, eval_set, label=""):
    """Run eval set through a RAG chain and collect results."""
    results_table = []
    
    print(f"\nEVALUATION: {label}")
    print("=" * 70)
    
    for i, item in enumerate(eval_set):
        result = chain.invoke({"query": item["question"]})
        
        # Check retrieval: did at least one source match the expected keyword?
        retrieved_sources = [doc.metadata.get("source", "") for doc in result["source_documents"]]
        retrieval_correct = any(item["expected_source_keyword"] in src for src in retrieved_sources)
        
        # Check faithfulness: does the answer reference context rather than hallucinate?
        answer = result["result"].lower()
        # A faithful answer should not say things completely unrelated to the retrieved chunks
        # Simple heuristic: check if answer doesn't claim lack of info when sources were found
        context_text = " ".join([doc.page_content.lower() for doc in result["source_documents"]])
        # Extract key terms from the answer and check if they appear in context
        faithful = not ("i don't have enough" in answer and retrieval_correct)
        if faithful:
            # Additional check: at least some key words from answer should be in context
            answer_words = set(answer.split()) - {"the", "a", "an", "is", "are", "was", "were", "in", "on", "at", "to", "for", "of", "and", "or", "that", "this", "it", "with", "by"}
            context_words = set(context_text.split())
            overlap = len(answer_words & context_words) / max(len(answer_words), 1)
            faithful = overlap > 0.2  # At least 20% of answer words should appear in context
        
        # Check correctness: does the answer align with expected?
        expected_lower = item["expected_answer"].lower()
        expected_key_terms = [term for term in expected_lower.split() if len(term) > 4]
        matching_terms = sum(1 for term in expected_key_terms if term in answer)
        correct = matching_terms >= len(expected_key_terms) * 0.3  # 30% key term overlap
        
        results_table.append({
            "question": item["question"],
            "expected": item["expected_answer"],
            "generated": result["result"],
            "sources": retrieved_sources,
            "retrieval": retrieval_correct,
            "faithfulness": faithful,
            "correctness": correct,
        })
        
        status_r = "✅" if retrieval_correct else "❌"
        status_f = "✅" if faithful else "❌"
        status_c = "✅" if correct else "❌"
        
        print(f"\nQ{i+1}: {item['question'][:60]}...")
        print(f"  Answer: {result['result'][:150]}...")
        print(f"  Retrieval: {status_r} | Faithfulness: {status_f} | Correctness: {status_c}")
    
    retrieval_score = sum(1 for r in results_table if r["retrieval"])
    faithfulness_score = sum(1 for r in results_table if r["faithfulness"])
    correctness_score = sum(1 for r in results_table if r["correctness"])
    
    print(f"\n{'='*70}")
    print(f"SCORES ({label}):")
    print(f"  Retrieval:    {retrieval_score}/5")
    print(f"  Faithfulness: {faithfulness_score}/5")
    print(f"  Correctness:  {correctness_score}/5")
    
    return results_table, {
        "retrieval": retrieval_score,
        "faithfulness": faithfulness_score,
        "correctness": correctness_score
    }

# Run primary evaluation
results_500, scores_500 = run_evaluation(rag_chain, eval_set, label="chunk_size=500")


EVALUATION: chunk_size=500



Q1: What architecture does DRQN use to handle partial observabil...
  Answer: DRQN uses a combination of a Long Short Term Memory (LSTM) and a Deep Q-Network to handle partial observability (arXiv:1507.06527v4). This architectur...
  Retrieval: ✅ | Faithfulness: ✅ | Correctness: ❌



Q2: What is the burn-in strategy in R2D2?...
  Answer: Based on the provided context, the burn-in strategy is described by its hypothesized benefit rather than its explicit procedural definition. The conte...
  Retrieval: ✅ | Faithfulness: ✅ | Correctness: ❌



Q3: What is the key contribution of Constrained Policy Optimizat...
  Answer: Based on the provided context, the key contribution of Constrained Policy Optimization (CPO) is that it is the **first general-purpose policy search a...
  Retrieval: ✅ | Faithfulness: ✅ | Correctness: ✅



Q4: What is the Gated Transformer-XL (GTrXL) and what problem do...
  Answer: Based on the provided context, the Gated Transformer-XL (GTrXL) is a proposed architecture that modifies the original Transformer and Transformer-XL v...
  Retrieval: ✅ | Faithfulness: ✅ | Correctness: ✅



Q5: How does DreamerV3 handle the challenge of varying signal ma...
  Answer: Based on the provided context, the text does not explicitly mention "varying signal magnitudes." However, it states that DreamerV3 uses "robustness te...
  Retrieval: ✅ | Faithfulness: ✅ | Correctness: ✅

SCORES (chunk_size=500):
  Retrieval:    5/5
  Faithfulness: 5/5
  Correctness:  3/5


---
## Stretch Goal A: Chunk Size Comparison

Running the full 5-question eval set against all three chunk sizes (300, 500, 1000).

In [17]:
# Build RAG chains for chunk_size=300 and chunk_size=1000
rag_chain_300 = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore_300.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": custom_prompt}
)

rag_chain_1000 = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore_1000.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": custom_prompt}
)

print("Built RAG chains for all 3 chunk sizes.")

Built RAG chains for all 3 chunk sizes.


In [18]:
# Run evaluation on chunk_size=300
results_300, scores_300 = run_evaluation(rag_chain_300, eval_set, label="chunk_size=300")


EVALUATION: chunk_size=300



Q1: What architecture does DRQN use to handle partial observabil...
  Answer: I don't have enough information in the provided context to answer this question. The context describes what DRQN does (handles partial observability, ...
  Retrieval: ✅ | Faithfulness: ❌ | Correctness: ❌



Q2: What is the burn-in strategy in R2D2?...
  Answer: Based on the provided context, the burn-in strategy involves allowing an RNN a certain number of steps to converge to a more "typical" state. This pro...
  Retrieval: ✅ | Faithfulness: ✅ | Correctness: ✅



Q3: What is the key contribution of Constrained Policy Optimizat...
  Answer: Based on the provided context from the paper "Constrained Policy Optimization," the key contribution is the proposal of Constrained Policy Optimizatio...
  Retrieval: ✅ | Faithfulness: ✅ | Correctness: ✅



Q4: What is the Gated Transformer-XL (GTrXL) and what problem do...
  Answer: Based on the provided context, the **Gated Transformer-XL (GTrXL)** is a proposed architecture designed to improve upon the canonical transformer.

It...
  Retrieval: ✅ | Faithfulness: ✅ | Correctness: ❌



Q5: How does DreamerV3 handle the challenge of varying signal ma...
  Answer: Based on the provided context, DreamerV3 handles the challenge of varying signal magnitudes by employing specific **robustness techniques**. The text ...
  Retrieval: ✅ | Faithfulness: ✅ | Correctness: ✅

SCORES (chunk_size=300):
  Retrieval:    5/5
  Faithfulness: 4/5
  Correctness:  3/5


In [19]:
# Run evaluation on chunk_size=1000
results_1000, scores_1000 = run_evaluation(rag_chain_1000, eval_set, label="chunk_size=1000")


EVALUATION: chunk_size=1000



Q1: What architecture does DRQN use to handle partial observabil...
  Answer: Based on the provided text from "Deep Recurrent Q-Learning for Partially Observable MDPs," DRQN handles partial observability by modifying the standar...
  Retrieval: ✅ | Faithfulness: ✅ | Correctness: ✅



Q2: What is the burn-in strategy in R2D2?...
  Answer: Based on the provided context, the burn-in strategy is described as a method to prevent "destructive updates" to the RNN parameters. These updates wou...
  Retrieval: ✅ | Faithfulness: ✅ | Correctness: ❌



Q3: What is the key contribution of Constrained Policy Optimizat...
  Answer: Based on the provided context from the paper "Constrained Policy Optimization," the key contribution is the proposal of Constrained Policy Optimizatio...
  Retrieval: ✅ | Faithfulness: ✅ | Correctness: ✅



Q4: What is the Gated Transformer-XL (GTrXL) and what problem do...
  Answer: Based on the provided context, the **Gated Transformer-XL (GTrXL)** is a novel architecture that modifies the original Transformer by reordering layer...
  Retrieval: ✅ | Faithfulness: ✅ | Correctness: ✅



Q5: How does DreamerV3 handle the challenge of varying signal ma...
  Answer: Based on the provided context, DreamerV3 handles the challenge of varying signal magnitudes across domains through specific **robustness techniques** ...
  Retrieval: ✅ | Faithfulness: ✅ | Correctness: ✅

SCORES (chunk_size=1000):
  Retrieval:    5/5
  Faithfulness: 5/5
  Correctness:  4/5


In [20]:
# Stretch Goal A: Comparison table
print("\nSTRETCH GOAL A: CHUNK SIZE COMPARISON")
print("=" * 60)
print(f"{'Metric':<20} {'chunk=300':>10} {'chunk=500':>10} {'chunk=1000':>10}")
print("-" * 60)
print(f"{'Retrieval':<20} {scores_300['retrieval']:>7}/5   {scores_500['retrieval']:>7}/5   {scores_1000['retrieval']:>7}/5")
print(f"{'Faithfulness':<20} {scores_300['faithfulness']:>7}/5   {scores_500['faithfulness']:>7}/5   {scores_1000['faithfulness']:>7}/5")
print(f"{'Correctness':<20} {scores_300['correctness']:>7}/5   {scores_500['correctness']:>7}/5   {scores_1000['correctness']:>7}/5")
print("=" * 60)

# Determine best
all_scores = {"300": scores_300, "500": scores_500, "1000": scores_1000}
best_chunk = max(all_scores, key=lambda k: sum(all_scores[k].values()))
print(f"\nBest performing chunk size: {best_chunk}")
print("\nStretch Goal A ✅")


STRETCH GOAL A: CHUNK SIZE COMPARISON
Metric                chunk=300  chunk=500 chunk=1000
------------------------------------------------------------
Retrieval                  5/5         5/5         5/5
Faithfulness               4/5         5/5         5/5
Correctness                3/5         3/5         4/5

Best performing chunk size: 1000

Stretch Goal A ✅


---
## Stretch Goal B: Hybrid Search Evaluation

Comparing pure vector search vs hybrid (BM25 + vector) using the same 5 eval questions.

In [21]:
# Build hybrid RAG chain
rag_chain_hybrid = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=ensemble_retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": custom_prompt}
)

# Run evaluation with hybrid search
results_hybrid, scores_hybrid = run_evaluation(rag_chain_hybrid, eval_set, label="Hybrid (BM25 + Vector)")


EVALUATION: Hybrid (BM25 + Vector)



Q1: What architecture does DRQN use to handle partial observabil...
  Answer: DRQN uses an architecture that combines a Long Short Term Memory (LSTM) with a Deep Q-Network. This allows the model to integrate information across f...
  Retrieval: ✅ | Faithfulness: ✅ | Correctness: ✅



Q2: What is the burn-in strategy in R2D2?...
  Answer: Based on the provided context, the burn-in strategy in R2D2 is defined by the following characteristics:

*   **Definition:** It involves unrolling th...
  Retrieval: ✅ | Faithfulness: ✅ | Correctness: ❌



Q3: What is the key contribution of Constrained Policy Optimizat...
  Answer: Based on the provided context, the key contribution of Constrained Policy Optimization (CPO) is that it is the **first general-purpose policy search a...
  Retrieval: ✅ | Faithfulness: ✅ | Correctness: ✅



Q4: What is the Gated Transformer-XL (GTrXL) and what problem do...
  Answer: Based on the provided context, the **Gated Transformer-XL (GTrXL)** is a novel architecture that modifies the original Transformer by reordering the l...
  Retrieval: ✅ | Faithfulness: ✅ | Correctness: ✅



Q5: How does DreamerV3 handle the challenge of varying signal ma...
  Answer: Based on the provided context, DreamerV3 handles the challenge of varying signal magnitudes across domains through the following methods:

*   **Robus...
  Retrieval: ✅ | Faithfulness: ✅ | Correctness: ✅

SCORES (Hybrid (BM25 + Vector)):
  Retrieval:    5/5
  Faithfulness: 5/5
  Correctness:  4/5


In [22]:
# Stretch Goal B: Comparison
print("\nSTRETCH GOAL B: VECTOR vs HYBRID COMPARISON")
print("=" * 50)
print(f"{'Metric':<20} {'Vector':>10} {'Hybrid':>10}")
print("-" * 50)
print(f"{'Retrieval':<20} {scores_500['retrieval']:>7}/5   {scores_hybrid['retrieval']:>7}/5")
print(f"{'Faithfulness':<20} {scores_500['faithfulness']:>7}/5   {scores_hybrid['faithfulness']:>7}/5")
print(f"{'Correctness':<20} {scores_500['correctness']:>7}/5   {scores_hybrid['correctness']:>7}/5")
print("=" * 50)

vector_total = sum(scores_500.values())
hybrid_total = sum(scores_hybrid.values())
if hybrid_total > vector_total:
    print("\n>> Hybrid search improved overall performance.")
elif hybrid_total == vector_total:
    print("\n>> Hybrid search performed the same as vector-only.")
else:
    print("\n>> Vector-only search outperformed hybrid for this dataset.")

print("\nStretch Goal B ✅")


STRETCH GOAL B: VECTOR vs HYBRID COMPARISON
Metric                   Vector     Hybrid
--------------------------------------------------
Retrieval                  5/5         5/5
Faithfulness               5/5         5/5
Correctness                3/5         4/5

>> Hybrid search improved overall performance.

Stretch Goal B ✅


---
## Save Evaluation Results

In [23]:
# Save all evaluation results to JSON
os.makedirs("eval", exist_ok=True)

eval_output = {
    "timestamp": datetime.now().isoformat(),
    "models": {
        "llm": "glm-5:cloud",
        "embeddings": "qwen3-embedding:8b (4096 dim)"
    },
    "scores": {
        "chunk_300": scores_300,
        "chunk_500": scores_500,
        "chunk_1000": scores_1000,
        "hybrid_500": scores_hybrid
    },
    "eval_questions": [
        {
            "question": item["question"],
            "expected_answer": item["expected_answer"],
            "generated_answer_500": results_500[i]["generated"],
            "retrieval_500": results_500[i]["retrieval"],
            "faithfulness_500": results_500[i]["faithfulness"],
            "correctness_500": results_500[i]["correctness"],
        }
        for i, item in enumerate(eval_set)
    ],
    "stretch_goals": {
        "A_chunk_comparison": True,
        "B_hybrid_search": True,
        "C_metadata_filtering": True,
        "D_streamlit_ui": "see app.py",
        "E_multi_document": "PDF + TXT + CSV"
    }
}

with open("eval/eval_results.json", "w") as f:
    json.dump(eval_output, f, indent=2, default=str)

print("Evaluation results saved to eval/eval_results.json")
print("\n" + "=" * 70)
print("ALL 6 STEPS + 5 STRETCH GOALS COMPLETE")
print("=" * 70)

Evaluation results saved to eval/eval_results.json

ALL 6 STEPS + 5 STRETCH GOALS COMPLETE
